# Differential expression of genes and antibodies and pathways

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import warnings 
import numpy as np
import pandas as pd
import seaborn as sns
import muon as mu
import anndata as ad
import gseapy as gp


warnings.filterwarnings('ignore')
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=200, facecolor='white')

In [ ]:
mdata = mu.read_h5mu("MDS_adata_for_publication_v2.h5mu")
mdata

In [ ]:
mdata['protein'].obs['patient_alias'] = mdata['rna'].obs['patient_alias'].copy()

In [ ]:
mdata['protein'].obs['leiden_rna'] = mdata['rna'].obs['leiden'].copy()

In [ ]:
mdata

In [ ]:
gene_set_names = gp.get_library_name(organism='Human')
gene_set_names

In [ ]:
HALLMARK_2024="data/MSigDB_HALLMARK_2024/h.all.v2024.1.Hs.symbols.gmt"

In [ ]:
def analyse_gsea(adata, group_name, uns_key, outfile_key):
    in_adata = adata


    #get DEG df
    deg_df = sc.get.rank_genes_groups_df(
        adata=in_adata,
        group=group_name,
        key=uns_key)
    
    #print("Now running prerank HALLMARK")
    #run GSEA pre rank
    #pre_rank = gp.prerank(deg_df.loc[:,['names', 'logfoldchanges']], gene_sets=[HALLMARK_2024])
    #pre_rank.res2d.to_csv(f"{outfile_key}_pre_rank_hallmark.csv")
    #pre_rank_term2 = pre_rank.res2d.Term
    #pre_rank.plot(terms=pre_rank_term2[:5], figsize=(7,7), ofname=f"{outfile_key}_prerank_hallmark.png")
    #plt.close()  

    #print("Now running prerank GO:BP")
    #run GSEA pre rank
    #pre_rank = gp.prerank(deg_df.loc[:,['names', 'logfoldchanges']], gene_sets=['GO_Biological_Process_2025'])
    #pre_rank.res2d.to_csv(f"{outfile_key}_pre_rank_GO.csv")
    #pre_rank_term2 = pre_rank.res2d.Term
    #pre_rank.plot(terms=pre_rank_term2[:5], figsize=(7,7), ofname=f"{outfile_key}_prerank_GO.png")
    #plt.close()  

    #run gsea enrich
    print("Now running enrich!")
    deg_sig = deg_df[deg_df['pvals_adj'] < 0.05]
    deg_up = deg_sig[deg_sig['logfoldchanges'] > 0]
    deg_down = deg_sig[deg_sig['logfoldchanges'] < 0]

    print("upregulated genes:",deg_up.shape)
    print("downregulated genes:",deg_down.shape)

    #Hallmark
    enr_up = gp.enrichr(gene_list = deg_up['names'], gene_sets=[HALLMARK_2024], verbose=True)
    enr_up.res2d.to_csv(f"{outfile_key}_enrich_up_hallmark.csv")

    enr_down = gp.enrichr(gene_list = deg_down['names'], gene_sets=[HALLMARK_2024], verbose=True)
    enr_down.res2d.to_csv(f"{outfile_key}_enrich_down_hallmark.csv")


    #plot gsea enrich

    enr_up.res2d['Direction'] = "Upregulated"
    enr_down.res2d['Direction'] = "Downregulated"

    enr_res = pd.concat([enr_up.res2d, enr_down.res2d])
    
    fig1, ax1 = plt.subplots(figsize=(7,7))
    ax1 = gp.dotplot(enr_res,x='Direction',
        title='HALLMARK 2024',  
        x_order = ["Upregulated","Downregulated"],
        size=3,
        show_ring=True,
        ax=ax1)

    plt.savefig(f"{outfile_key}_enrich_dotplot_hallmark.png", dpi=200, bbox_inches = "tight")
    plt.close()

    fig2, ax2 = plt.subplots(figsize=(7,7))
    ax2 = gp.barplot(enr_res, x='Direction',
        group='Direction',
        title='HALLMARK 2024',  
        color=['b','r'],
        ax=ax2)

    plt.savefig(f"{outfile_key}_enrich_barplot_hallmark.png", dpi=200, bbox_inches = "tight")
    plt.close()

    #GO:BP
    enr_up = gp.enrichr(gene_list = deg_up['names'], gene_sets=['GO_Biological_Process_2025'], verbose=True)
    enr_up.res2d.to_csv(f"{outfile_key}_enrich_up_GO.csv")

    enr_down = gp.enrichr(gene_list = deg_down['names'], gene_sets=['GO_Biological_Process_2025'], verbose=True)
    enr_down.res2d.to_csv(f"{outfile_key}_enrich_down_GO.csv")


    #plot gsea enrich

    enr_up.res2d['Direction'] = "Upregulated"
    enr_down.res2d['Direction'] = "Downregulated"

    enr_res = pd.concat([enr_up.res2d, enr_down.res2d])
    
    fig1, ax1 = plt.subplots(figsize=(7,7))
    ax1 = gp.dotplot(enr_res,x='Direction',
        title='GO:BP 2025',  
        x_order = ["Upregulated","Downregulated"],
        size=3,
        show_ring=True,
        ax=ax1)

    plt.savefig(f"{outfile_key}_enrich_dotplot_GO.png", dpi=200, bbox_inches = "tight")
    plt.close()

    fig2, ax2 = plt.subplots(figsize=(7,7))
    ax2 = gp.barplot(enr_res, x='Direction',
        group='Direction',
        title='GO:BP 2025',  
        color=['b','r'],
        ax=ax2)

    plt.savefig(f"{outfile_key}_enrich_barplot_GO.png", dpi=200, bbox_inches = "tight")
    plt.close()



In [ ]:
var_df = mdata.mod['rna'].var.copy()
var_df = var_df["chromosome"]

In [ ]:
def get_merged_expr_df(adata, group1, group2, df, obs_key="leiden"):

    df1 = df
    
    X1 = adata[adata.obs[obs_key] == group1].X.toarray()

    mean_expr = X1.mean(axis=0)
    frac_expr = (X1 > 0).sum(axis=0) / X1.shape[0]


    X2 = adata[adata.obs[obs_key] == group2].X.toarray()

    mean_expr2 = X2.mean(axis=0)
    frac_expr2 = (X2 > 0).sum(axis=0) / X2.shape[0]

    #epsilon = 1e-6
    #log2foldchange = np.log2((np.expm1(mean_expr) + epsilon)/ (np.expm1(mean_expr2) + epsilon))
    #lnfoldchange = np.log((np.expm1(mean_expr) + epsilon)/ (np.expm1(mean_expr2) + epsilon))

    exp_df = pd.DataFrame({'names': adata.var_names, 
        f'{group1}_mean_expr': mean_expr, 
        f'{group1}_frac_expr': frac_expr,
        f'{group2}_mean_expr': mean_expr2,
        f'{group2}_frac_expr':frac_expr2})
        #f'log2FC':log2foldchange,
        #f'lnFC':lnfoldchange})

    exp_df = exp_df.merge(var_df, left_on="names", right_index=True, how="left")
    exp_df.index = exp_df.index.astype(str)
    
    #exp_df_sorted = exp_df.set_index("names").loc[df1["names"]].reset_index()

    merged_df = df1.merge(exp_df, on="names", how="left")
    
    return merged_df

## P17

In [ ]:
p17 = mdata[mdata.obs["rna:patient_alias"] == "P17"]
p17

In [ ]:
#remove RP and MT genes

mask = ~p17.mod['rna'].var_names.str.startswith(("RP", "MT"))
p17.mod['rna'] = p17.mod['rna'][:, mask].copy()
p17

### cluster K vs L

In [ ]:
sc.tl.rank_genes_groups(p17['rna'], 
    'celltype_v3', 
    groups=['Atypical cluster K','Atypical cluster L'], 
    reference="Atypical cluster L",
    method='wilcoxon', 
    use_raw=False,
    key_added="K_vs_L_rank_genes_groups")

In [ ]:
df1 = sc.get.rank_genes_groups_df(p17['rna'], group='Atypical cluster K',key='K_vs_L_rank_genes_groups')
get_merged_expr_df(p17['rna'], "Atypical cluster K", "Atypical cluster L", df1).to_csv("data/P17_KvsL_Diff_GEX.csv")


In [ ]:
sc.tl.rank_genes_groups(p17['protein'], 
    'celltype_v3', 
    groups=['Atypical cluster K','Atypical cluster L'], 
    reference="Atypical cluster L",
    method='wilcoxon', 
    use_raw=False,
    key_added="K_vs_L_rank_genes_groups")

sc.get.rank_genes_groups_df(p17['protein'], group='Atypical cluster K', key='K_vs_L_rank_genes_groups').to_csv("data/P17_KvsL_Diff_ADT.csv")

In [ ]:
sc.queries.enrich(p17['rna'],
    "Atypical cluster K", 
    key="K_vs_L_rank_genes_groups", 
    gprofiler_kwargs={"no_evidences":False,
    "sources":['KEGG','REAC']},
    log2fc_min=0.25, 
    pval_cutoff=0.05).to_csv("data/P17_KvsL_Diff_pathway.csv")

In [ ]:
analyse_gsea(p17['rna'], "Atypical cluster K", "K_vs_L_rank_genes_groups", "data/P17_KvsL")

### cluster M vs L

In [ ]:
sc.tl.rank_genes_groups(p17['rna'], 
    'celltype_v3', 
    groups=['Atypical cluster M','Atypical cluster L'], 
    reference="Atypical cluster L",
    method='wilcoxon', 
    use_raw=False,
    key_added="M_vs_L_rank_genes_groups")

df1 = sc.get.rank_genes_groups_df(p17['rna'], group='Atypical cluster M', key='M_vs_L_rank_genes_groups')
get_merged_expr_df(p17['rna'], "Atypical cluster M", "Atypical cluster L", df1).to_csv("data/P17_MvsL_Diff_GEX.csv")

In [ ]:
sc.tl.rank_genes_groups(p17['protein'], 
    'celltype_v3', 
    groups=['Atypical cluster M','Atypical cluster L'], 
    reference="Atypical cluster L",
    method='wilcoxon', 
    use_raw=False,
    key_added="M_vs_L_rank_genes_groups")

sc.get.rank_genes_groups_df(p17['protein'], group='Atypical cluster M', key='M_vs_L_rank_genes_groups').to_csv("data/P17_MvsL_Diff_ADT.csv")

In [ ]:
sc.queries.enrich(p17['rna'],"Atypical cluster M", key="M_vs_L_rank_genes_groups", 
    gprofiler_kwargs={"no_evidences":False,
    "sources":['KEGG','REAC']},
    log2fc_min=0.25, 
    pval_cutoff=0.05).to_csv("data/P17_MvsL_Diff_pathway.csv")

In [ ]:
analyse_gsea(p17['rna'], "Atypical cluster M", "M_vs_L_rank_genes_groups", "data/P17_MvsL")

### M vs K

In [ ]:
sc.tl.rank_genes_groups(p17['rna'], 
    'celltype_v3', 
    groups=['Atypical cluster M','Atypical cluster K'], 
    reference="Atypical cluster K",
    method='wilcoxon', 
    use_raw=False,
    key_added="M_vs_K_rank_genes_groups")

df1 = sc.get.rank_genes_groups_df(p17['rna'], group='Atypical cluster M', key='M_vs_K_rank_genes_groups')
get_merged_expr_df(p17['rna'], "Atypical cluster M", "Atypical cluster K", df1).to_csv("data/P17_MvsK_Diff_GEX.csv")

In [ ]:
sc.tl.rank_genes_groups(p17['protein'], 
    'celltype_v3', 
    groups=['Atypical cluster M','Atypical cluster K'], 
    reference="Atypical cluster K",
    method='wilcoxon', 
    use_raw=False,
    key_added="M_vs_K_rank_genes_groups")

sc.get.rank_genes_groups_df(p17['protein'], group='Atypical cluster M', key='M_vs_K_rank_genes_groups').to_csv("data/P17_MvsK_Diff_ADT.csv")

In [ ]:
sc.queries.enrich(p17['rna'],"Atypical cluster M", key="M_vs_K_rank_genes_groups", 
    gprofiler_kwargs={"no_evidences":False,
    "sources":['KEGG','REAC']},
    log2fc_min=0.25, 
    pval_cutoff=0.05).to_csv("data/P17_MvsK_Diff_pathway.csv")

In [ ]:
analyse_gsea(p17['rna'], "Atypical cluster M", "M_vs_K_rank_genes_groups", "data/P17_MvsK")

## P18

In [ ]:
p18 = mdata[mdata.obs["rna:patient_alias"] == "P18"].copy()
mask18 = ~p18.mod['rna'].var_names.str.startswith(("RP", "MT"))
p18.mod['rna'] = p18.mod['rna'][:, mask18].copy()
p18

### Cluster A vs B

In [ ]:
sc.tl.rank_genes_groups(p18['rna'], 
    'celltype_v3', 
    groups=['Atypical cluster A','Atypical cluster B'], 
    reference="Atypical cluster B",
    method='wilcoxon', 
    use_raw=False,
    key_added="A_vs_B_rank_genes_groups")

df1 = sc.get.rank_genes_groups_df(p18['rna'], group='Atypical cluster A', key='A_vs_B_rank_genes_groups')
get_merged_expr_df(p18['rna'], "Atypical cluster A", "Atypical cluster B", df1).to_csv("data/P18_AvsB_Diff_GEX.csv")

In [ ]:
sc.tl.rank_genes_groups(p18['protein'], 
    'celltype_v3', 
    groups=['Atypical cluster A','Atypical cluster B'], 
    reference="Atypical cluster B",
    method='wilcoxon', 
    use_raw=False,
    key_added="A_vs_B_rank_genes_groups")

sc.get.rank_genes_groups_df(p18['protein'], group='Atypical cluster A', key='A_vs_B_rank_genes_groups').to_csv("data/P18_AvsB_Diff_ADT.csv")

In [ ]:
sc.queries.enrich(p18['rna'],"Atypical cluster A", key="A_vs_B_rank_genes_groups", gprofiler_kwargs={"no_evidences":False,"sources":['KEGG','REAC']}).to_csv("data/P18_AvsB_Diff_pathway.csv")

In [ ]:
analyse_gsea(p18['rna'], "Atypical cluster A", "A_vs_B_rank_genes_groups", "data/P18_AvsB")

### Cluster C vs B

In [ ]:
sc.tl.rank_genes_groups(p18['rna'], 
    'celltype_v3', 
    groups=['Atypical cluster C','Atypical cluster B'], 
    reference="Atypical cluster B",
    method='wilcoxon', 
    use_raw=False,
    key_added="C_vs_B_rank_genes_groups")

df1 = sc.get.rank_genes_groups_df(p18['rna'], group='Atypical cluster C', key='C_vs_B_rank_genes_groups')
get_merged_expr_df(p18['rna'], "Atypical cluster C", "Atypical cluster B", df1).to_csv("data/P18_CvsB_Diff_GEX.csv")

In [ ]:
sc.tl.rank_genes_groups(p18['protein'], 
    'celltype_v3', 
    groups=['Atypical cluster C','Atypical cluster B'], 
    reference="Atypical cluster B",
    method='wilcoxon', 
    use_raw=False,
    key_added="C_vs_B_rank_genes_groups")

sc.get.rank_genes_groups_df(p18['protein'], group='Atypical cluster C', key='C_vs_B_rank_genes_groups').to_csv("data/P18_CvsB_Diff_ADT.csv")

In [ ]:
sc.queries.enrich(p18['rna'],"Atypical cluster C", key="C_vs_B_rank_genes_groups", gprofiler_kwargs={"no_evidences":False,"sources":['KEGG','REAC']}).to_csv("data/P18_CvsB_Diff_pathway.csv")

In [ ]:
analyse_gsea(p18['rna'], "Atypical cluster C", "C_vs_B_rank_genes_groups", "data/P18_CvsB")

### A vs C

In [ ]:
sc.tl.rank_genes_groups(p18['rna'], 
    'celltype_v3', 
    groups=['Atypical cluster A','Atypical cluster C'], 
    reference="Atypical cluster C",
    method='wilcoxon', 
    use_raw=False,
    key_added="A_vs_C_rank_genes_groups")

df1 = sc.get.rank_genes_groups_df(p18['rna'], group='Atypical cluster A', key='A_vs_C_rank_genes_groups')
get_merged_expr_df(p18['rna'], "Atypical cluster A", "Atypical cluster C", df1).to_csv("data/P18_AvsC_Diff_GEX.csv")

In [ ]:
sc.tl.rank_genes_groups(p18['protein'], 
    'celltype_v3', 
    groups=['Atypical cluster A','Atypical cluster C'], 
    reference="Atypical cluster C",
    method='wilcoxon', 
    use_raw=False,
    key_added="A_vs_C_rank_genes_groups")

sc.get.rank_genes_groups_df(p18['protein'], group='Atypical cluster A', key='A_vs_C_rank_genes_groups').to_csv("data/P18_AvsC_Diff_ADT.csv")

In [ ]:
sc.queries.enrich(p18['rna'],"Atypical cluster A", key="A_vs_C_rank_genes_groups", gprofiler_kwargs={"no_evidences":False,"sources":['KEGG','REAC']}).to_csv("data/P18_AvsC_Diff_pathway.csv")

In [ ]:
analyse_gsea(p18['rna'], "Atypical cluster A", "A_vs_C_rank_genes_groups", "data/P18_AvsC")

## P02

In [ ]:
p02 = mdata[mdata.obs["rna:patient_alias"] == "P02"].copy()
mask02 = ~p02.mod['rna'].var_names.str.startswith(("RP", "MT"))
p02.mod['rna'] = p02.mod['rna'][:, mask02].copy()
p02

In [ ]:
sc.tl.rank_genes_groups(p02['rna'], 
    'celltype_v3', 
    groups=['Atypical cluster G','Atypical cluster H'], 
    reference="Atypical cluster H",
    method='wilcoxon', 
    use_raw=False,
    key_added="G_vs_H_rank_genes_groups")

df1 = sc.get.rank_genes_groups_df(p02['rna'], group='Atypical cluster G', key='G_vs_H_rank_genes_groups')
get_merged_expr_df(p02['rna'], "Atypical cluster G", "Atypical cluster H", df1).to_csv("data/P02_GvsH_Diff_GEX.csv")

In [ ]:
sc.tl.rank_genes_groups(p02['protein'], 
    'celltype_v3', 
    groups=['Atypical cluster G','Atypical cluster H'], 
    reference="Atypical cluster H",
    method='wilcoxon', 
    use_raw=False,
    key_added="G_vs_H_rank_genes_groups")

sc.get.rank_genes_groups_df(p02['protein'], group='Atypical cluster G', key='G_vs_H_rank_genes_groups').to_csv("data/P02_GvsH_Diff_ADT.csv")

In [ ]:
sc.queries.enrich(p02['rna'],"Atypical cluster G", key="G_vs_H_rank_genes_groups", gprofiler_kwargs={"no_evidences":False,"sources":['KEGG','REAC']}).to_csv("data/P02_GvsH_Diff_pathway.csv")

In [ ]:
analyse_gsea(p02['rna'], "Atypical cluster G", "G_vs_H_rank_genes_groups", "data/P02_GvsH")

## P08

In [ ]:
p08 = mdata[mdata.obs["rna:patient_alias"] == "P08"].copy()
mask08 = ~p08.mod['rna'].var_names.str.startswith(("RP", "MT"))
p08.mod['rna'] = p08.mod['rna'][:, mask08].copy()
p08

###  N vs I

In [ ]:
sc.tl.rank_genes_groups(p08['rna'], 
    'celltype_v3', 
    groups=['Atypical cluster N','Atypical cluster I'], 
    reference="Atypical cluster I",
    method='wilcoxon', 
    use_raw=False,
    key_added="N_vs_I_rank_genes_groups")

df1 = sc.get.rank_genes_groups_df(p08['rna'], group='Atypical cluster N', key='N_vs_I_rank_genes_groups')
get_merged_expr_df(p08['rna'], "Atypical cluster N", "Atypical cluster I", df1).to_csv("data/P08_NvsI_Diff_GEX.csv")

In [ ]:
sc.tl.rank_genes_groups(p08['protein'], 
    'celltype_v3', 
    groups=['Atypical cluster N','Atypical cluster I'], 
    reference="Atypical cluster I",
    method='wilcoxon', 
    use_raw=False,
    key_added="N_vs_I_rank_genes_groups")

sc.get.rank_genes_groups_df(p08['protein'], group='Atypical cluster N', key='N_vs_I_rank_genes_groups').to_csv("data/P08_NvsI_Diff_ADT.csv")

In [ ]:
sc.queries.enrich(p08['rna'],"Atypical cluster N", key="N_vs_I_rank_genes_groups", gprofiler_kwargs={"no_evidences":False,"sources":['KEGG','REAC']}).to_csv("data/P08_NvsI_Diff_pathway.csv")

In [ ]:
analyse_gsea(p08['rna'], "Atypical cluster N", "N_vs_I_rank_genes_groups", "data/P08_NvsI")

### K vs I

In [ ]:
sc.tl.rank_genes_groups(p08['rna'], 
    'celltype_v3', 
    groups=['Atypical cluster K','Atypical cluster I'], 
    reference="Atypical cluster I",
    method='wilcoxon', 
    use_raw=False,
    key_added="K_vs_I_rank_genes_groups")

df1 = sc.get.rank_genes_groups_df(p08['rna'], group='Atypical cluster K', key='K_vs_I_rank_genes_groups')
get_merged_expr_df(p08['rna'], "Atypical cluster K", "Atypical cluster I", df1).to_csv("data/P08_KvsI_Diff_GEX.csv")

In [ ]:
sc.tl.rank_genes_groups(p08['protein'], 
    'celltype_v3', 
    groups=['Atypical cluster K','Atypical cluster I'], 
    reference="Atypical cluster I",
    method='wilcoxon', 
    use_raw=False,
    key_added="K_vs_I_rank_genes_groups")

sc.get.rank_genes_groups_df(p08['protein'], group='Atypical cluster K', key='K_vs_I_rank_genes_groups').to_csv("data/P08_KvsI_Diff_ADT.csv")

In [ ]:
sc.queries.enrich(p08['rna'],"Atypical cluster K", key="K_vs_I_rank_genes_groups", gprofiler_kwargs={"no_evidences":False,"sources":['KEGG','REAC']}).to_csv("data/P08_KvsI_Diff_pathway.csv")

In [ ]:
analyse_gsea(p08['rna'], "Atypical cluster K", "K_vs_I_rank_genes_groups", "data/P08_KvsI")

### K vs N

In [ ]:
sc.tl.rank_genes_groups(p08['rna'], 
    'celltype_v3', 
    groups=['Atypical cluster K','Atypical cluster N'], 
    reference="Atypical cluster N",
    method='wilcoxon', 
    use_raw=False,
    key_added="K_vs_N_rank_genes_groups")

df1 = sc.get.rank_genes_groups_df(p08['rna'], group='Atypical cluster K', key='K_vs_N_rank_genes_groups')
get_merged_expr_df(p08['rna'], "Atypical cluster K", "Atypical cluster N", df1).to_csv("data/P08_KvsN_Diff_GEX.csv")

In [ ]:
sc.tl.rank_genes_groups(p08['protein'], 
    'celltype_v3', 
    groups=['Atypical cluster K','Atypical cluster N'], 
    reference="Atypical cluster N",
    method='wilcoxon', 
    use_raw=False,
    key_added="K_vs_N_rank_genes_groups")

sc.get.rank_genes_groups_df(p08['protein'], group='Atypical cluster K', key='K_vs_N_rank_genes_groups').to_csv("data/P08_KvsN_Diff_ADT.csv")

In [ ]:
sc.queries.enrich(p08['rna'],"Atypical cluster K", key="K_vs_N_rank_genes_groups", gprofiler_kwargs={"no_evidences":False,"sources":['KEGG','REAC']}).to_csv("data/P08_KvsN_Diff_pathway.csv")

In [ ]:
analyse_gsea(p08['rna'], "Atypical cluster K", "K_vs_N_rank_genes_groups", "data/P08_KvsN")

## P03

In [ ]:
p03 = mdata[mdata.obs["rna:patient_alias"] == "P03"].copy()
mask03 = ~p03.mod['rna'].var_names.str.startswith(("RP", "MT"))
p03.mod['rna'] = p03.mod['rna'][:, mask03].copy()
p03

### D vs GMP

In [ ]:
sc.tl.rank_genes_groups(p03['rna'], 
    'celltype_v3', 
    groups=['Atypical cluster D','GMP'], 
    reference="GMP",
    method='wilcoxon', 
    use_raw=False,
    key_added="D_vs_GMP_rank_genes_groups")

df1 = sc.get.rank_genes_groups_df(p03['rna'], group='Atypical cluster D', key='D_vs_GMP_rank_genes_groups')
get_merged_expr_df(p03['rna'], "Atypical cluster D", "GMP", df1).to_csv("data/P03_DvsGMP_Diff_GEX.csv")

In [ ]:
sc.tl.rank_genes_groups(p03['protein'], 
    'celltype_v3', 
    groups=['Atypical cluster D','GMP'], 
    reference="GMP",
    method='wilcoxon', 
    use_raw=False,
    key_added="D_vs_GMP_rank_genes_groups")

sc.get.rank_genes_groups_df(p03['protein'], group='Atypical cluster D', key='D_vs_GMP_rank_genes_groups').to_csv("data/P03_DvsGMP_Diff_ADT.csv")

In [ ]:
sc.queries.enrich(p03['rna'],"Atypical cluster D", key="D_vs_GMP_rank_genes_groups", gprofiler_kwargs={"no_evidences":False,"sources":['KEGG','REAC']}).to_csv("data/P03_DvsGMP_Diff_pathway.csv")

In [ ]:
analyse_gsea(p03['rna'], "Atypical cluster D", "D_vs_GMP_rank_genes_groups", "data/P03_DvsGMP")

### D vs M

In [ ]:
sc.tl.rank_genes_groups(p03['rna'], 
    'celltype_v3', 
    groups=['Atypical cluster D','Atypical cluster M'], 
    reference="Atypical cluster M",
    method='wilcoxon', 
    use_raw=False,
    key_added="D_vs_M_rank_genes_groups")

df1 = sc.get.rank_genes_groups_df(p03['rna'], group='Atypical cluster D', key='D_vs_M_rank_genes_groups')
get_merged_expr_df(p03['rna'], "Atypical cluster D", "Atypical cluster M", df1).to_csv("data/P03_DvsM_Diff_GEX.csv")

In [ ]:
sc.tl.rank_genes_groups(p03['protein'], 
    'celltype_v3', 
    groups=['Atypical cluster D','Atypical cluster M'], 
    reference="Atypical cluster M",
    method='wilcoxon', 
    use_raw=False,
    key_added="D_vs_M_rank_genes_groups")

sc.get.rank_genes_groups_df(p03['protein'], group='Atypical cluster D', key='D_vs_M_rank_genes_groups').to_csv("data/P03_DvsM_Diff_ADT.csv")

In [ ]:
sc.queries.enrich(p03['rna'],"Atypical cluster D", key="D_vs_M_rank_genes_groups", gprofiler_kwargs={"no_evidences":False,"sources":['KEGG','REAC']}).to_csv("data/P03_DvsM_Diff_pathway.csv")

In [ ]:
analyse_gsea(p03['rna'], "Atypical cluster D", "D_vs_M_rank_genes_groups", "data/P03_DvsM")

## P09

In [ ]:
p09 = mdata[mdata.obs["rna:patient_alias"] == "P09"].copy()
mask09 = ~p09.mod['rna'].var_names.str.startswith(("RP", "MT"))
p09.mod['rna'] = p09.mod['rna'][:, mask09].copy()
p09

In [ ]:
p09['protein'].obs['celltype_v3'].value_counts()

In [ ]:
sc.tl.rank_genes_groups(p09['rna'], 
    'celltype_v3', 
    groups=['HSC_P09','MPP'], 
    reference="MPP",
    method='wilcoxon', 
    use_raw=False,
    key_added="HSC_vs_MPP_rank_genes_groups")

df1 = sc.get.rank_genes_groups_df(p09['rna'], group='HSC_P09', key='HSC_vs_MPP_rank_genes_groups')
get_merged_expr_df(p09['rna'], "HSC_P09", "MPP", df1).to_csv("data/P09_HSCvsMPP_Diff_GEX.csv")

In [ ]:
sc.tl.rank_genes_groups(p09['protein'], 
    'celltype_v3', 
    groups=['HSC','MPP'], 
    reference="MPP",
    method='wilcoxon', 
    use_raw=False,
    key_added="HSC_vs_MPP_rank_genes_groups")

sc.get.rank_genes_groups_df(p09['protein'], group='HSC', key='HSC_vs_MPP_rank_genes_groups').to_csv("data/P09_HSCvsMPP_Diff_ADT.csv")

In [ ]:
sc.queries.enrich(p09['rna'],"HSC_P09", key="HSC_vs_MPP_rank_genes_groups", gprofiler_kwargs={"no_evidences":False,"sources":['KEGG','REAC']}).to_csv("data/P09_HSCvsMPP_Diff_pathway.csv")

In [ ]:
analyse_gsea(p09['rna'], "HSC_P09", "HSC_vs_MPP_rank_genes_groups", "data/P09_HSCvsMPP")

## P11

In [ ]:
p11 = mdata[mdata.obs["rna:patient_alias"] == "P11"].copy()
mask11 = ~p11.mod['rna'].var_names.str.startswith(("RP", "MT"))
p11.mod['rna'] = p11.mod['rna'][:, mask11].copy()
p11

In [ ]:
sc.tl.rank_genes_groups(p11['rna'], 
    'celltype_v3', 
    groups=['Atypical cluster E','Monocyte progenitor'], 
    reference="Monocyte progenitor",
    method='wilcoxon', 
    use_raw=False,
    key_added="E_vs_Monocyte_rank_genes_groups")

df1 = sc.get.rank_genes_groups_df(p11['rna'], group='Atypical cluster E', key='E_vs_Monocyte_rank_genes_groups')
get_merged_expr_df(p11['rna'], "Atypical cluster E", "Monocyte progenitor", df1).to_csv("data/P11_EvsMonocyte_Diff_GEX.csv")

In [ ]:
sc.queries.enrich(p11['rna'],"Atypical cluster E", key="E_vs_Monocyte_rank_genes_groups", gprofiler_kwargs={"no_evidences":False,"sources":['KEGG','REAC']}).to_csv("data/P11_EvsMonocyte_Diff_pathway.csv")

In [ ]:
analyse_gsea(p11['rna'], "Atypical cluster E", "E_vs_Monocyte_rank_genes_groups", "data/P11_EvsMonocyte")

## P12

In [ ]:
p12 = mdata[mdata.obs["rna:patient_alias"] == "P12"].copy()
mask12 = ~p12.mod['rna'].var_names.str.startswith(("RP", "MT"))
p12.mod['rna'] = p12.mod['rna'][:, mask12].copy()
p12

### HSC vs MPP

In [ ]:
sc.tl.rank_genes_groups(p12['rna'], 
    'celltype_v3', 
    groups=['HSC','MPP'], 
    reference="MPP",
    method='wilcoxon', 
    use_raw=False,
    key_added="HSC_vs_MPP_rank_genes_groups")

df1 = sc.get.rank_genes_groups_df(p12['rna'], group='HSC', key='HSC_vs_MPP_rank_genes_groups')
get_merged_expr_df(p12['rna'], "HSC", "MPP", df1).to_csv("data/P12_HSCvsMPP_Diff_GEX.csv")

In [ ]:
sc.tl.rank_genes_groups(p12['protein'], 
    'celltype_v3', 
    groups=['HSC','MPP'], 
    reference="MPP",
    method='wilcoxon', 
    use_raw=False,
    key_added="HSC_vs_MPP_rank_genes_groups")

sc.get.rank_genes_groups_df(p12['protein'], group='HSC', key='HSC_vs_MPP_rank_genes_groups').to_csv("data/P12_HSCvsMPP_Diff_ADT.csv")

In [ ]:
sc.queries.enrich(p12['rna'],"HSC", key="HSC_vs_MPP_rank_genes_groups", gprofiler_kwargs={"no_evidences":False,"sources":['KEGG','REAC']}).to_csv("data/P12_HSCvsMPP_Diff_pathway.csv")

In [ ]:
analyse_gsea(p12['rna'], "HSC", "HSC_vs_MPP_rank_genes_groups", "data/P12_HSCvsMPP")

### HSC vs MEP/MKP

In [ ]:
sc.tl.rank_genes_groups(p12['rna'], 
    'celltype_v3', 
    groups=['HSC','MEP/MKP'], 
    reference="MEP/MKP",
    method='wilcoxon', 
    use_raw=False,
    key_added="HSC_vs_MEP/MKP_rank_genes_groups")

df1 = sc.get.rank_genes_groups_df(p12['rna'], group='HSC', key='HSC_vs_MEP/MKP_rank_genes_groups')
get_merged_expr_df(p12['rna'], "HSC", "MEP/MKP", df1).to_csv("data/P12_HSCvsMEP_MKP_Diff_GEX.csv")

In [ ]:
sc.tl.rank_genes_groups(p12['protein'], 
    'celltype_v3', 
    groups=['HSC','MEP/MKP'], 
    reference="MEP/MKP",
    method='wilcoxon', 
    use_raw=False,
    key_added="HSC_vs_MEP/MKP_rank_genes_groups")

sc.get.rank_genes_groups_df(p12['protein'], group='HSC', key='HSC_vs_MEP/MKP_rank_genes_groups').to_csv("data/P12_HSCvsMEP_MKP_Diff_ADT.csv")

In [ ]:
sc.queries.enrich(p12['rna'],"HSC", key="HSC_vs_MEP/MKP_rank_genes_groups", gprofiler_kwargs={"no_evidences":False,"sources":['KEGG','REAC']}).to_csv("data/P12_HSCvsMEP_MKP_Diff_pathway.csv")

In [ ]:
analyse_gsea(p12['rna'], "HSC", "HSC_vs_MEP/MKP_rank_genes_groups", "data/P12_HSCvsMEP_MKP")

### F vs M

In [ ]:
sc.tl.rank_genes_groups(p12['rna'], 
    'celltype_v3', 
    groups=['Atypical cluster F','Atypical cluster M'], 
    reference="Atypical cluster M",
    method='wilcoxon', 
    use_raw=False,
    key_added="F_vs_M_rank_genes_groups")

df1 = sc.get.rank_genes_groups_df(p12['rna'], group='Atypical cluster F', key='F_vs_M_rank_genes_groups')
get_merged_expr_df(p12['rna'], "Atypical cluster F", "Atypical cluster M", df1).to_csv("data/P12_FvsM_Diff_GEX.csv")

In [ ]:
#Cannot do ADT differential expression as there is not CITE-seq data for cluster F in patient P12

#sc.tl.rank_genes_groups(p12['protein'], 
#    'celltype_v3', 
#    groups=['Atypical cluster F','Atypical cluster M'], 
#    reference="Atypical cluster M",
#    method='wilcoxon', 
#    use_raw=False,
#    key_added="F_vs_M_rank_genes_groups")

#sc.get.rank_genes_groups_df(p12['protein'], group='Atypical cluster F', key='F_vs_M_rank_genes_groups').to_csv("data/P12_FvsM_Diff_ADT.csv")

In [ ]:
sc.queries.enrich(p12['rna'],"Atypical cluster F", key="F_vs_M_rank_genes_groups", gprofiler_kwargs={"no_evidences":False,"sources":['KEGG','REAC']}).to_csv("data/P12_FvsM_Diff_pathway.csv")

In [ ]:
analyse_gsea(p12['rna'], "Atypical cluster F", "F_vs_M_rank_genes_groups", "data/P12_FvsM")

## P01

In [ ]:
p01 = mdata[mdata.obs["rna:patient_alias"] == "P01"].copy()
mask01 = ~p01.mod['rna'].var_names.str.startswith(("RP", "MT"))
p01.mod['rna'] = p01.mod['rna'][:, mask01].copy()
p01

### J vs N

In [ ]:
sc.tl.rank_genes_groups(p01['rna'], 
    'celltype_v3', 
    groups=['Atypical cluster J','Atypical cluster N'], 
    reference="Atypical cluster N",
    method='wilcoxon', 
    use_raw=False,
    key_added="J_vs_N_rank_genes_groups")

df1 = sc.get.rank_genes_groups_df(p01['rna'], group='Atypical cluster J', key='J_vs_N_rank_genes_groups')
get_merged_expr_df(p01['rna'], "Atypical cluster J", "Atypical cluster N", df1).to_csv("data/P01_JvsN_Diff_GEX.csv")

In [ ]:
sc.tl.rank_genes_groups(p01['protein'], 
    'celltype_v3', 
    groups=['Atypical cluster J','Atypical cluster N'], 
    reference="Atypical cluster N",
    method='wilcoxon', 
    use_raw=False,
    key_added="J_vs_N_rank_genes_groups")

sc.get.rank_genes_groups_df(p01['protein'], group='Atypical cluster J', key='J_vs_N_rank_genes_groups').to_csv("data/P01_JvsN_Diff_ADT.csv")

In [ ]:
sc.queries.enrich(p01['rna'],"Atypical cluster J", key="J_vs_N_rank_genes_groups", gprofiler_kwargs={"no_evidences":False,"sources":['KEGG','REAC']}).to_csv("data/P01_JvsN_Diff_pathway.csv")

In [ ]:
analyse_gsea(p01['rna'], "Atypical cluster J", "J_vs_N_rank_genes_groups", "data/P01_JvsN")

## Sensitive vs Resistant

In [ ]:
all = mdata.copy()
maskall = ~all.mod['rna'].var_names.str.startswith(("RP", "MT"))
all.mod['rna'] = all.mod['rna'][:, maskall].copy()
all

In [ ]:
sensitive = ["Atypical cluster J",
             "Atypical cluster D",
             "Atypical cluster E",
             "Atypical cluster F",
             "Atypical cluster K",
             "Atypical cluster A",
             "Atypical cluster G",]

resistant = ["Atypical cluster N",
             "Atypical cluster B",
             "Atypical cluster C",
             "Atypical cluster H",
             "Atypical cluster L",
             "Atypical cluster I",
             "Atypical cluster M",]

In [ ]:
all['rna'].obs['sensitive_resistant'] = ""
all['rna'].obs.loc[all['rna'].obs["celltype_v3"].isin(sensitive), "sensitive_resistant"] = "sensitive"
all['rna'].obs.loc[all['rna'].obs["celltype_v3"].isin(resistant), "sensitive_resistant"] = "resistant"


In [ ]:
all['rna'].obs[["sensitive_resistant", "celltype_v3"]]

In [ ]:
sc.tl.rank_genes_groups(all['rna'], 
    'sensitive_resistant', 
    groups=['sensitive','resistant'], 
    reference="resistant",
    method='wilcoxon', 
    use_raw=False,
    key_added="sensitive_vs_resistant_rank_genes_groups")

df1 = sc.get.rank_genes_groups_df(all['rna'], group='sensitive', key='sensitive_vs_resistant_rank_genes_groups')
get_merged_expr_df(all['rna'], "sensitive", "resistant", df1, obs_key="sensitive_resistant").to_csv("data/all_sensitivevsresistant_Diff_GEX.csv")

In [ ]:
all['protein'].obs['sensitive_resistant'] = all['rna'].obs['sensitive_resistant'].copy()

In [ ]:
#sc.tl.rank_genes_groups(all['protein'], 
#    'sensitive_resistant', 
#    groups=['sensitive','resistant'], 
#    reference="resistant",
#    method='wilcoxon', 
#    use_raw=False,
#    key_added="sensitive_vs_resistant_rank_genes_groups")

df1 = sc.get.rank_genes_groups_df(all['protein'], group='sensitive', key='sensitive_vs_resistant_rank_genes_groups')
get_merged_expr_df(all['protein'], "sensitive", "resistant", df1, obs_key="sensitive_resistant").to_csv("data/all_sensitivevsresistant_Diff_ADT.csv")

In [ ]:
sc.queries.enrich(all['rna'],"sensitive", key="sensitive_vs_resistant_rank_genes_groups", gprofiler_kwargs={"no_evidences":False,"sources":['KEGG','REAC']}).to_csv("data/all_sensitivevsresistant_Diff_pathway.csv")

In [ ]:
analyse_gsea(all['rna'], "sensitive", "sensitive_vs_resistant_rank_genes_groups", "data/all_sensitivevsresistant")

Hi Priscilla, can you please run an extra DEG comparison for P01? You previously ran J vs N (leiden 3 vs leiden 14). Could you please now run J (leiden 3) vs leiden 23 (previously classified as an MPP). I only need the hallmarks pathway analysis, no need to do the GO terms. 

Could you please also run leiden 14 vs leiden 23 for P01?

## P01 cluster 3 vs cluster 23

In [ ]:
p01 = mdata[mdata.obs["rna:patient_alias"] == "P01"].copy()

In [ ]:
mask01 = ~p01.mod['rna'].var_names.str.startswith(("RP", "MT"))
p01.mod['rna'] = p01.mod['rna'][:, mask01].copy()
p01

In [ ]:
p01['rna'].obs['leiden'].value_counts()

In [ ]:
sc.tl.rank_genes_groups(p01['rna'], 
    'leiden', 
    groups=['3','23'], 
    reference="23",
    method='wilcoxon', 
    use_raw=False,
    key_added="3_vs_23_rank_genes_groups")

df1 = sc.get.rank_genes_groups_df(p01['rna'], group='3', key='3_vs_23_rank_genes_groups')
get_merged_expr_df(p01['rna'], "3", "23", df1).to_csv("data/P01_3vs23_Diff_GEX.csv")

In [ ]:
sc.tl.rank_genes_groups(p01['protein'], 
    'leiden_rna', 
    groups=['3','23'], 
    reference="23",
    method='wilcoxon', 
    use_raw=False,
    key_added="3_vs_23_rank_genes_groups")

sc.get.rank_genes_groups_df(p01['protein'], group='3', key='3_vs_23_rank_genes_groups').to_csv("data/P01_3vs23_Diff_ADT.csv")

In [ ]:
analyse_gsea(p01['rna'], "3", "3_vs_23_rank_genes_groups", "data/P01_3vs23")

## 14 vs 23

In [ ]:
sc.tl.rank_genes_groups(p01['rna'], 
    'leiden', 
    groups=['14','23'], 
    reference="23",
    method='wilcoxon', 
    use_raw=False,
    key_added="14_vs_23_rank_genes_groups")

df1 = sc.get.rank_genes_groups_df(p01['rna'], group='14', key='14_vs_23_rank_genes_groups')
get_merged_expr_df(p01['rna'], "14", "23", df1).to_csv("data/P01_14vs23_Diff_GEX.csv")

In [ ]:
sc.tl.rank_genes_groups(p01['protein'], 
    'leiden_rna', 
    groups=['14','23'], 
    reference="23",
    method='wilcoxon', 
    use_raw=False,
    key_added="14_vs_23_rank_genes_groups")

sc.get.rank_genes_groups_df(p01['protein'], group='14', key='14_vs_23_rank_genes_groups').to_csv("data/P01_14vs23_Diff_ADT.csv")

In [ ]:
analyse_gsea(p01['rna'], "14", "14_vs_23_rank_genes_groups", "data/P01_14vs23")

# MISC